# RAG - Question Anwsering Git

## Git Integration

In [1]:
import requests
import os

from google.colab import userdata

github_token = userdata.get('GITHUB_TOKEN')

username = 'bivar'

# Get information about a user
user_url = f'https://api.github.com/users/{username}'
headers = {'Authorization': f'token {github_token}'}

response = requests.get(user_url, headers=headers)

if response.status_code == 200:
    user_data = response.json()
    print(f"User: {user_data['login']}")
    print(f"Name: {user_data['name']}")
    print(f"Public Repos: {user_data['public_repos']}")

    # Get all repositories for the user
    repos_url = user_data['repos_url']
    repos_response = requests.get(repos_url, headers=headers)

    repo_codes = []
    if repos_response.status_code == 200:
        repos_data = repos_response.json()
        print("\nRepositories:")
        for repo in repos_data:
            print(f"\n- {repo['name']}")
            # Get contents of each repository (only root level for simplicity)
            contents_url = f"https://api.github.com/repos/{username}/{repo['name']}/contents"
            contents_response = requests.get(contents_url, headers=headers)
            if contents_response.status_code == 200:
                contents_data = contents_response.json()
                for item in contents_data:
                    if item['type'] == 'file':
                        file_url =  item['download_url']
                        file_response = requests.get(file_url, headers=headers)
                        if file_response.status_code == 200:
                            repo_codes.append({"name" : item['name'],
                                               "content" : file_response.text,
                                               "repository" : repo['name']})
                        else:
                            print(f"      Error getting file content: {file_response.status_code}")
                            pass



            else:
                print(f"  Error getting contents: {contents_response.status_code}")
                print(contents_response.json())
    else:
        print(f"Error getting repositories: {repos_response.status_code}")
        print(repos_response.json())

else:
    print(f"Error getting user data: {response.status_code}")
    print(response.json())

User: bivar
Name: Beca
Public Repos: 19

Repositories:

- bivar

- CG_ProjectFinal-Test-

- CorneV3Basic

- CVRP_APA

- data-related-studies

- desafio-6-2020

- file-name-shortener

- hands-on-ml-scikit

- llm_studies

- next_level_week

- numerico

- parallel-grid-search

- recommendationSystem

- RecoTour

- sheets-scripts

- SIGReal

- sklearn_transforms

- text-summarization

- Twitter-thread-mining


In [2]:
import pandas as pd
pd.DataFrame(repo_codes)

,name,content,repository
0,README.md,\n# Rebeca Bivar (bec/becky/beca)\n## Sobre mi...,bivar
1,.gitignore,# Prerequisites\n*.d\n\n# Compiled Object file...,CG_ProjectFinal-Test-
2,README.md,# My custom Corne Layout\n\nthis is my custom ...,CorneV3Basic
3,config.h,/*\nThis is the c configuration file for the k...,CorneV3Basic
4,keymap.c,/*\nCopyright 2019 @foostan\nCopyright 2020 Dr...,CorneV3Basic
5,rules.mk,MOUSEKEY_ENABLE = no # Mouse keys\nRGBLI...,CorneV3Basic
6,.gitattributes,# Auto detect text files and perform LF normal...,CVRP_APA
7,LICENSE,MIT License\n\nCopyright (c) 2019 bivar\n\nPer...,CVRP_APA
8,Pvris_Analysis.ipynb,"{\n ""nbformat"": 4,\n ""nbformat_minor"": 0,\n ...",data-related-studies
9,README.md,# data-related-studies,data-related-studies


## Vector DB

In [30]:
#!pip install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.4/128.4 kB 9.7 MB/s eta 

In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize ChromaDB client
client = chromadb.Client()
# Load a pre-trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create a collection (or get an existing one)
collection_name = "github_repo_codes"
try:
    collection = client.create_collection(name=collection_name)
    print(f"Collection '{collection_name}' created.")
except:
    collection = client.get_collection(name=collection_name)
    print(f"Collection '{collection_name}' already exists.")

# Prepare data for insertion into Chroma
documents = [item['content'] for item in repo_codes]
metadatas = [{"name": item['name'], "repository": item['repository']} for item in repo_codes]
ids = [f"file_{i}" for i in range(len(repo_codes))]

# Generate embeddings
embeddings = model.encode(documents).tolist()

# Add data to the collection
collection.add(
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Added {len(repo_codes)} documents to the collection.")

Collection 'github_repo_codes' created.


In [6]:
# Verify the number of documents in the collection
count = collection.count()
print(f"Number of documents in the collection: {count}")

# You can also retrieve some data to verify
# result = collection.get(ids=["file_0", "file_1"], include=['metadatas', 'documents'])
# print("\nExample retrieved data:")
# print(result)

Number of documents in the collection: 49


## RAG

### Load the hugging face model


In [7]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained sentence transformer model
query_model = SentenceTransformer('all-MiniLM-L6-v2')

### Generate query embedding

Using the loaded Hugging Face model to generate an embedding for the user's query.


In [16]:
# Define the user query
query = "Show me code related to Twitter API"

# Generate the embedding for the query
query_embedding = query_model.encode(query).tolist()

### Perform similarity search

Now we are going to perform a similarity search on the Chroma vector database to find relevant code snippets.


In [17]:
N_RESULTS = 3

# Perform similarity search
results = collection.query(
    query_embeddings=query_embedding,
    n_results=N_RESULTS,
    include=['documents', 'metadatas']
)

# Print the results
print(results)

{'ids': [['file_46', 'file_47', 'file_48']], 'embeddings': None, 'documents': [['# Twitter-thread-mining\nMining really long Twitter messages-threads\n\nThis codebase is useful to explore twitter threads for which no direct api is avalibale in any python twitter wrapper\n\nCommand to run is \n```\n$python3 findAllTweetsInThread.py\n```\n\nTo know all tweets of a user; use\n```\n$python twitterAllTweetsOfUser.py\n```\nFor more information please explore my medium blog\n```\nhttps://medium.com/@lih.verma/mining-really-long-twitter-messages-threads-569d42bc0e1c\n```\n', '\nimport tweepy\nimport json\n# Enter your keys/secrets as strings in the following fields\ncredentials = {}\ncredentials[\'CONSUMER_KEY\'] = \'\'\ncredentials[\'CONSUMER_SECRET\'] = \'\'\ncredentials[\'ACCESS_TOKEN\'] = \'\'\ncredentials[\'ACCESS_SECRET\'] = \'\'\n\nauth = tweepy.OAuthHandler(credentials.get(\'CONSUMER_KEY\'), credentials.get(\'CONSUMER_SECRET\'))\nauth.set_access_token(credentials.get(\'ACCESS_TOKEN\'),

### Retrieve relevant documents

Extract the relevant code snippets and their associated metadata (repository and file name) from the search results.


In [18]:
retrieved_documents = []

for i in range(len(results['documents'][0])):
    document = results['documents'][0][i]
    metadata = results['metadatas'][0][i]
    name = metadata['name']
    repository = metadata['repository']
    retrieved_documents.append({"document": document, "name": name, "repository": repository})

print(retrieved_documents)

[{'document': '# Twitter-thread-mining\nMining really long Twitter messages-threads\n\nThis codebase is useful to explore twitter threads for which no direct api is avalibale in any python twitter wrapper\n\nCommand to run is \n```\n$python3 findAllTweetsInThread.py\n```\n\nTo know all tweets of a user; use\n```\n$python twitterAllTweetsOfUser.py\n```\nFor more information please explore my medium blog\n```\nhttps://medium.com/@lih.verma/mining-really-long-twitter-messages-threads-569d42bc0e1c\n```\n', 'name': 'README.md', 'repository': 'Twitter-thread-mining'}, {'document': '\nimport tweepy\nimport json\n# Enter your keys/secrets as strings in the following fields\ncredentials = {}\ncredentials[\'CONSUMER_KEY\'] = \'\'\ncredentials[\'CONSUMER_SECRET\'] = \'\'\ncredentials[\'ACCESS_TOKEN\'] = \'\'\ncredentials[\'ACCESS_SECRET\'] = \'\'\n\nauth = tweepy.OAuthHandler(credentials.get(\'CONSUMER_KEY\'), credentials.get(\'CONSUMER_SECRET\'))\nauth.set_access_token(credentials.get(\'ACCESS_T

### Synthesize the answer

Use the retrieved code snippets and metadata to formulate an answer to the user's query, indicating which repositories contain the requested content.


In [20]:
# Initialize an empty string to store the synthesized answer
synthesized_answer = f"Based on your query ({query}), I found the following relevant code snippets in these repositories:\n\n"

# Iterate through the retrieved documents
for doc_info in retrieved_documents:
    repository = doc_info['repository']
    name = doc_info['name']
    document = doc_info['document']

    # Taking the first 100 characters of the document for brevity
    snippet_preview = document[:100] + "..." if len(document) > 100 else document
    synthesized_answer += f"- Repository: {repository}, File: {name}\n\nSnippet: \n{snippet_preview}\n\n"
    synthesized_answer += "\n------------------------------------------------------------------------\n"

# Print the final synthesized answer
print(synthesized_answer)

Based on your query (Show me code related to Twitter API), I found the following relevant code snippets in these repositories:

- Repository: Twitter-thread-mining, File: README.md

Snippet: 
# Twitter-thread-mining
Mining really long Twitter messages-threads

This codebase is useful to expl...


------------------------------------------------------------------------
- Repository: Twitter-thread-mining, File: findAllTweetsInThread.py

Snippet: 

import tweepy
import json
# Enter your keys/secrets as strings in the following fields
credentials ...


------------------------------------------------------------------------
- Repository: Twitter-thread-mining, File: twitterAllTweetsOfUser.py

Snippet: 

import tweepy
import json
import csv
# Enter your keys/secrets as strings in the following fields
c...


------------------------------------------------------------------------

